In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import display

def plot_3d_beams(beam_angle_deg, rov_altitude, light_spacing, front_back_spacing, forward_tilt_deg, side_tilt_deg):
    fig = plt.figure(figsize=(6, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.set_box_aspect([1, 1, 1])

    theta = np.radians(beam_angle_deg / 2)
    beam_radius = np.tan(theta) * rov_altitude

    # Light positions (front left, front right, back left, back right)
    x_positions = [-light_spacing / 2, light_spacing / 2]
    y_positions = [-front_back_spacing / 2, front_back_spacing / 2]
    light_labels = {
        (-1,  1): "FL",
        ( 1,  1): "FR",
        (-1, -1): "BL",
        ( 1, -1): "BR"
    }

    for y in y_positions:
        for x in x_positions:
            # Determine light label
            label_key = (int(np.sign(x)), int(np.sign(y)))
            label = light_labels.get(label_key, "")

            # Determine tilt
            forward_tilt = np.radians(forward_tilt_deg if y > 0 else -forward_tilt_deg)
            side_tilt = np.radians(side_tilt_deg if x > 0 else -side_tilt_deg)

            # Beam direction
            dz = -1
            dx = np.tan(side_tilt)
            dy = np.tan(forward_tilt)
            norm = np.sqrt(dx**2 + dy**2 + dz**2)
            dx, dy, dz = dx / norm, dy / norm, dz / norm

            # Beam end at seafloor
            distance = rov_altitude / -dz
            beam_center_x = x + dx * distance
            beam_center_y = y + dy * distance
            beam_center_z = 0

            # Beam circle footprint
            n = 30
            angles = np.linspace(0, 2 * np.pi, n)
            circle_x = beam_center_x + beam_radius * np.cos(angles)
            circle_y = beam_center_y + beam_radius * np.sin(angles)
            circle_z = np.zeros_like(circle_x)

            verts = [list(zip(circle_x, circle_y, circle_z))]
            ax.plot([x, beam_center_x], [y, beam_center_y], [rov_altitude, beam_center_z], color='gray')
            ax.add_collection3d(Poly3DCollection(verts, alpha=0.1, facecolor='deepskyblue'))

            # Add label above the light
            ax.text(x, y, rov_altitude + 0.05, label, color='black', ha='center', va='bottom', fontsize=10)

    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_zlim(0, 1.25)
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_zlabel("Z (m)")
    ax.set_title("3D ROV Light Beam Spread (with Tilt and Labels)")

    plt.tight_layout()
    plt.show()

widgets.interact(
    plot_3d_beams,
    beam_angle_deg=widgets.FloatSlider(value=75, min=10, max=135, step=1, description='Beam Angle (°)'),
    rov_altitude=widgets.FloatSlider(value=0.8, min=0.1, max=2.0, step=0.05, description='ROV Altitude (m)'),
    light_spacing=widgets.FloatSlider(value=0.514, min=0.1, max=2.0, step=0.01, description='Light Spacing (m)'),
    front_back_spacing=widgets.FloatSlider(value=0.4635, min=0.1, max=1.5, step=0.01, description='Front/Back Spacing (m)'),
    forward_tilt_deg=widgets.FloatSlider(value=10, min=0, max=45, step=1, description='Forward Tilt (°)'),
    side_tilt_deg=widgets.FloatSlider(value=10, min=0, max=45, step=1, description='Side Tilt (°)')
)
